# Metabolite enumeration rules

How the SMIRKS transformation rules were built and validated, and the ceiling they put on
enumerate-then-rank.

**Question:** if plausible metabolites are enumerated by rules and a model ranks them, how often
is the right answer in the list, and how long is the list?

In [1]:
import sys, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# Imported from src/evaluation.py so the rules and the metrics agree on molecule identity.
from src.evaluation import canonical, standardize

def load(name):
    return pd.read_csv(ROOT / "data" / name, dtype=str, keep_default_na=False, na_values=[])

train = pd.concat([load(f) for f in ["drugbank.csv", "gloryx.csv", "metxbiodb.csv"]], ignore_index=True)
test = load("test_set.csv")
print(f"train rows {len(train):,}   test rows {len(test):,}")


train rows 7,487   test rows 309


## 1. The rules

The rules are stored in `rules/metabolism_rules.csv`. For a product containing disconnected
fragments, `keep_largest_fragment` retains its largest connected fragment. RDKit can also return
several separate products from one reaction; those remain separate candidates. The checked
product limitations are documented in `docs/FINDINGS.md`, section 21.

In [2]:
rules_df = pd.read_csv(ROOT / "rules" / "metabolism_rules.csv")
print(f"{len(rules_df)} rules\n")
print(rules_df["reaction_class"].value_counts().to_string())
rules_df.head(8)

42 rules

reaction_class
phase I: oxidation       15
phase II: conjugation    15
phase I: reduction        4
phase I: dealkylation     4
phase I: hydrolysis       4


,name,reaction_class,smirks,keep_largest_fragment
0,aromatic C-H hydroxylation,phase I: oxidation,[cH:1]>>[c:1]O,0
1,aliphatic C-H hydroxylation,phase I: oxidation,"[CX4;H1,H2,H3:1]>>[C:1]O",0
2,N-oxidation,phase I: oxidation,[NX3;H0;!$(N=*);!$(N[!#6;!#1]):1]>>[N+:1][O-],0
3,S-oxidation,phase I: oxidation,[SX2;!$(S=O):1]>>[S:1]=O,0
4,S-dioxidation,phase I: oxidation,[SX2;!$(S=O):1]>>O=[S:1]=O,0
5,CH2 -> C=O,phase I: oxidation,[CX4;H2:1]>>[C:1]=O,0
6,CH-OH -> C=O,phase I: oxidation,[CX4;H1:1][OX2H1]>>[C:1]=O,0
7,primary alcohol -> acid,phase I: oxidation,[CX4;H2:1][OX2H1]>>[C:1](=O)O,0


## 2. Applying a rule

A SMIRKS is find-and-replace on a molecule. `[cH:1]>>[c:1]O` means "find an aromatic carbon
carrying a hydrogen, replace it with that carbon carrying an OH". RDKit applies it at **every**
matching position, so one rule on one molecule gives several candidates.

In [3]:
COMPILED = []
for _, r in rules_df.iterrows():
    rxn = AllChem.ReactionFromSmarts(r["smirks"])
    rxn.Initialize()
    COMPILED.append((r["name"], rxn, bool(r["keep_largest_fragment"])))
print(f"{len(COMPILED)}/{len(rules_df)} rules compiled")

def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
    return max(frags, key=lambda f: f.GetNumHeavyAtoms()) if len(frags) > 1 else mol

def enumerate_metabolites(parent_smiles, rules=None, per_rule_cap=200):
    """Apply every rule at every matching site. Returns {candidate_smiles: rule_that_made_it}."""
    mol = Chem.MolFromSmiles(parent_smiles)
    if mol is None:
        return {}
    out = {}
    for name, rxn, keep_largest in (rules or COMPILED):
        try:
            products = rxn.RunReactants((mol,))
        except Exception:
            continue
        n = 0
        for tup in products:
            for p in tup:
                try:
                    if keep_largest:
                        p = largest_fragment(p)
                    Chem.SanitizeMol(p)
                    smi = Chem.MolToSmiles(p, canonical=True)
                except Exception:
                    continue
                if smi and smi not in out:
                    out[smi] = name
                n += 1
                if n > per_rule_cap:
                    break
            if n > per_rule_cap:
                break
    return out

42/42 rules compiled


## 3. Comparing molecules

The source data uses different tautomer representations, including `C(O)=N` and `C(=O)N`.
On a sample of 1,200 training pairs, 58% of parents change under RDKit tautomer canonicalisation.

Comparisons use `standardize()`: normalise functional groups, neutralise charges, choose a
canonical tautomer and remove stereochemistry. The resulting metric measures recovery under
that identity policy. Training targets, predictions and references use the same comparison key.

Multiple stereoisomers are recorded for 22.6% of distinct training skeletons. Those forms can
reflect different enzymes or conditions. Removing stereo merges five test pairs, but that count
does not measure its effect on prediction accuracy. The cached ranker scores 33.7% with exact
stereo-sensitive matching and 46.9% with full standardisation (`docs/FINDINGS.md`, section 8).

In [4]:
# standardize(): normalise, neutralise, canonical tautomer, drop stereo. canonical(): plain RDKit.

checks = [
    ("amide vs imidic acid", "CC(=O)Nc1ccccc1", "CC(O)=Nc1ccccc1"),
    ("lactam vs lactim", "O=C1CCCCN1", "OC1=NCCCC1"),
    ("2-pyridone", "O=c1cccc[nH]1", "Oc1ccccn1"),
    ("charged vs neutral", "CC(=O)[O-]", "CC(=O)O"),
    ("keto vs enol", "CC(=O)CC(=O)C", "CC(O)=CC(=O)C"),
]
pd.DataFrame(
    [(n, canonical(a) == canonical(b), standardize(a) == standardize(b)) for n, a, b in checks],
    columns=["case", "plain SMILES match", "standardized match"],
)


,case,plain SMILES match,standardized match
0,amide vs imidic acid,False,True
1,lactam vs lactim,False,True
2,2-pyridone,False,True
3,charged vs neutral,False,True
4,keto vs enol,False,True


## 4. Worked example: bupropion

The holdout test set lists bupropion six times, once per known metabolite, all +16 Da. Two differ
only in stereochemistry, so after standardising there are **five distinct molecules**.

In [5]:
bupropion = canonical("C[C@H](NC(C)(C)C)C(=O)c1cccc(Cl)c1")
is_bupropion = test["parent_smiles"].map(canonical) == bupropion
known = {standardize(s) for s in test.loc[is_bupropion, "metabolite_smiles"]}
candidates = enumerate_metabolites(bupropion)

print(f"parent                        : {bupropion}")
print(f"known metabolites in test set : {len(known)}")
print(f"candidates generated          : {len(candidates)}")
print(f"of those, known metabolites   : {sum(1 for c in candidates if standardize(c) in known)}")

pd.DataFrame(
    [
        (rule, smi, "YES" if standardize(smi) in known else "")
        for smi, rule in sorted(candidates.items(), key=lambda kv: kv[1])
    ],
    columns=["rule that produced it", "candidate", "is a known metabolite"],
)

parent                        : C[C@H](NC(C)(C)C)C(=O)c1cccc(Cl)c1
known metabolites in test set : 5
candidates generated          : 19


of those, known metabolites   : 4

,rule that produced it,candidate,is a known metabolite
0,GSH conjugation (halide),C[C@H](NC(C)(C)C)C(=O)c1cccc(SC[C@H](NC(=O)CC[...,
1,N-acetylation,CC(=O)N([C@@H](C)C(=O)c1cccc(Cl)c1)C(C)(C)C,
2,N-dealkylation (any alkyl),CC(C)(C)N,
3,N-dealkylation (any alkyl),CCC(=O)c1cccc(Cl)c1,
4,N-dealkylation (any alkyl),C[C@H](N)C(=O)c1cccc(Cl)c1,
5,N-dealkylation (any alkyl),CC(C)C,
6,N-glucuronidation,C[C@@H](C(=O)c1cccc(Cl)c1)N([C@@H]1O[C@@H](C(=...,
7,N-hydroxylation,C[C@@H](C(=O)c1cccc(Cl)c1)N(O)C(C)(C)C,
8,N-methylation,C[C@@H](C(=O)c1cccc(Cl)c1)N(C)C(C)(C)C,
9,N-sulfation,C[C@@H](C(=O)c1cccc(Cl)c1)N(C(C)(C)C)S(=O)(=O)O,


One of the five references is the cyclised morpholinol. The current single-round rule set
does not produce it. Enumeration-depth experiments are recorded in `docs/FINDINGS.md`, section 11.

## 5. Coverage: is the answer in the list, and how long is the list

If the true metabolite is not in the candidate set, no ranker can recover it, so coverage is the
ceiling on enumerate-then-rank.

In [6]:
test_eval = test.copy()
test_eval["parent"] = test_eval["parent_smiles"].map(canonical)
test_eval["exact"] = test_eval["metabolite_smiles"].map(canonical)
test_eval["nostereo"] = [canonical(s, keep_stereo=False) for s in test_eval["metabolite_smiles"]]
test_eval["standard"] = test_eval["metabolite_smiles"].map(standardize)

cache = {}
rows = []
for parent, m_exact, m_nostereo, m_standard in zip(
    test_eval["parent"], test_eval["exact"], test_eval["nostereo"], test_eval["standard"]
):
    if parent not in cache:
        cands = enumerate_metabolites(parent)
        cache[parent] = (
            set(cands),
            {canonical(c, keep_stereo=False) for c in cands},
            {standardize(c) for c in cands},
            len(cands),
        )
    exact_set, nostereo_set, standard_set, n = cache[parent]
    rows.append((m_exact in exact_set, m_nostereo in nostereo_set, m_standard in standard_set, n))

cov = pd.DataFrame(rows, columns=["exact", "stereo_stripped", "standardized", "n_candidates"])
sizes = cov["n_candidates"].values

print(f"test pairs {len(cov)}   unique parents {len(cache)}\n")
print(pd.DataFrame({
    "match criterion": ["exact canonical SMILES", "stereochemistry stripped", "standardized"],
    "coverage": [f"{cov[c].mean():.1%}" for c in ["exact", "stereo_stripped", "standardized"]],
}).to_string(index=False))
print(f"\ncandidates per parent: median {np.median(sizes):.0f}  mean {sizes.mean():.1f}  "
      f"p90 {np.percentile(sizes, 90):.0f}  max {sizes.max()}")
stereo_cost = cov["stereo_stripped"].mean() - cov["exact"].mean()
tautomer_cost = cov["standardized"].mean() - cov["stereo_stripped"].mean()
print(f"\nstereochemistry accounts for {stereo_cost:.1%} of the test set")
print(f"additional standardization accounts for a further {tautomer_cost:.1%}")

test pairs 309   unique parents 155

         match criterion coverage
  exact canonical SMILES    53.7%
stereochemistry stripped    68.6%
            standardized    72.2%

candidates per parent: median 31  mean 33.2  p90 50  max 105

stereochemistry accounts for 14.9% of the test set
tautomer convention accounts for a further 3.6%


With standardised matching, the candidate lists contain the reference for 72.2% of test pairs.
The median is 32 candidates per parent. A five-slot output can recover at most 71.5%, because
two parents have six distinct reachable answers.

MetaTrans reports achieved top-five recall of 42.5% on a different cohort (Chem. Sci. 2020,
Table 4). Coverage and achieved recall measure different parts of the prediction pipeline.

## 6. Per-rule positive controls

Running each rule in isolation distinguishes rules that produce no matching references from
rules whose products were attributed to an earlier rule. Seven rules with zero attributed hits
in the combined analysis matched references when tested alone.

This checks firing and reference recovery. Chemical validation of every generated product is a
separate check; known product defects are listed in `docs/FINDINGS.md`, section 21.

In [7]:
sample = train.drop_duplicates(subset=["parent_smiles", "metabolite_smiles"]).sample(200, random_state=0)
sample_pairs = [
    (canonical(p), standardize(m))
    for p, m in zip(sample["parent_smiles"], sample["metabolite_smiles"])
]

solo = []
for name, rxn, keep in COMPILED:
    fired = explains = 0
    for parent, reference in sample_pairs:
        cands = enumerate_metabolites(parent, rules=[(name, rxn, keep)])
        if cands:
            fired += 1
            if reference in {standardize(c) for c in cands}:
                explains += 1
    solo.append((name, fired, explains))

solo_df = pd.DataFrame(solo, columns=["rule", "fires on N parents", "explains N pairs"])
solo_df = solo_df.sort_values("explains N pairs", ascending=False)
dead = solo_df[solo_df["fires on N parents"] == 0]
print(f"rules that did not fire on this 200-pair sample: {len(dead)}")
if len(dead):
    print(dead["rule"].to_string(index=False))
solo_df.head(20)

rules that never fire at all (these would be genuinely broken): 0


,rule,fires on N parents,explains N pairs
27,O-glucuronidation,123,36
1,aliphatic C-H hydroxylation,180,31
19,N-dealkylation (any alkyl),88,18
26,glycoside/ether cleavage,73,17
20,O-dealkylation (any alkyl),67,14
0,aromatic C-H hydroxylation,158,12
30,O-sulfation,123,6
29,acid glucuronidation,27,5
14,ketone -> alcohol,100,4
12,dehydrogenation C-C,117,3


## 7. Remaining misses

In [8]:
missed = test_eval[~cov["standardized"].values]

def atom_delta(p, q):
    a, b = Chem.MolFromSmiles(p), Chem.MolFromSmiles(q)
    if a is None or b is None:
        return "?"
    ca = Counter(x.GetSymbol() for x in Chem.AddHs(a).GetAtoms())
    cb = Counter(x.GetSymbol() for x in Chem.AddHs(b).GetAtoms())
    parts = [
        f"{k}{cb.get(k, 0) - ca.get(k, 0):+d}"
        for k in sorted(set(ca) | set(cb))
        if cb.get(k, 0) != ca.get(k, 0)
    ]
    return " ".join(parts) or "(none)"

deltas = Counter(atom_delta(p, m) for p, m in zip(missed["parent"], missed["exact"]))
singletons = sum(1 for v in deltas.values() if v == 1)

print(f"still missed            : {len(missed)} / {len(test_eval)} ({len(missed)/len(test_eval):.1%})")
print(f"distinct reaction types : {len(deltas)}")
print(f"of which singletons     : {singletons} "
      f"({singletons/len(deltas):.0%} of types, {singletons/len(missed):.0%} of misses)")
top_delta, top_count = deltas.most_common(1)[0]
print(f"largest remaining group : {top_delta} with {top_count} pairs "
      f"({top_count / len(test_eval):.1%} of test)\n")

pd.DataFrame(deltas.most_common(10), columns=["atom delta", "missed pairs"])

still missed            : 86 / 309 (27.8%)
distinct reaction types : 60
of which singletons     : 50 (83% of types, 58% of misses)
largest remaining group : O+1 with 11 pairs (3.6% of test)



,atom delta,missed pairs
0,O+1,11
1,C-1 H-2,5
2,H-2,4
3,(none),3
4,H-2 O+1,3
5,C-1 O-2,2
6,C-2 H-4 O-2,2
7,H+2 O+1,2
8,C-1 H-5 N-1 O+2,2
9,C-3 H-4,2


The 86 uncovered references span 60 formula-change families. Fifty families occur once, and
the largest contains 11 pairs, or 3.6% of the test set. The five largest families total 26 pairs,
or 8.4 points of potential coverage. Recovering them would require working rules at the correct
sites; the resulting recall would also depend on ranking the enlarged candidate lists.

This analysis inspected test references while deciding where rule development should stop.
The resulting evaluation is exploratory and needs confirmation on a newly reserved cohort.

Pooling generated candidates with rule candidates increased reachability and reduced achieved
recall in the recorded experiments (`docs/FINDINGS.md`, sections 13 and 19).

Exact and stereo-stripped rule coverage differ by 14.9 points. Conjugation templates have fixed
stereochemistry that can disagree with the reference annotations. Standardised matching removes
that distinction, so the headline endpoint does not assess stereochemical accuracy.